In [ ]:
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from pathlib import Path
import re

plt.rcParams.update({
    "font.size" : 23,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def read_col_csv(filePath, colIdx, skipHeader):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx, delimiter=',')
    f.close()
    return vec

def get_probe_coord_value(filePath, coordIdx):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordValue = float(header.split('(')[1].split(')')[0].split(',')[coordIdx])
    return coordValue

def calculate_average(lists):
    return [sum(values) / len(lists) for values in zip(*lists)]

In [ ]:
x = 1.06
# x = 1.54
# x = 2.02
U0 = 0.039

folderVec = [
    # "/Users/maggie/repo/scripts/cases/flow_past_cylinder_Re3900/3900_cml_smag017_40dx_intpbb/probes",
    # "/Users/maggie/repo/scripts/cases/flow_past_cylinder_Re3900/3900_cml_smag017_40dx_intpbb/3d_inflow",
    # "/Users/maggie/repo/scripts/cases/flow_past_cylinder_Re3900/3900_cml_smag017_40dx_intpbb/3d_inflow",
    "/Users/maggie/repo/scripts/cases/flow_past_cylinder_Re3900/3900_cml_smag017_64dx_intpbb/a800_inflow"
]
print(folderVec)

labelVec = [
    # 'LBM Cumulant Outflow',
    # 'LBM Cumulant Inflow (20k-100k)',
    # 'LBM Cumulant Inflow (100k-200k)',
    # "LBM Cumulant (D=40dx EquOut)",
    # "LBM Cumulant (D=64dx EquOut)"
    "Present LBM"
]

avgRgVec = [
    # [20000, 60000],
    # [20000, 100000],
    # [100000, 200000],
    [100000, 200000],
]

# yVec = np.linspace(-1.5, 1.2, 10)
# zVec = np.linspace(-1.5, 1.5, 101)
# print(yVec)

yVecVec = [
    # np.linspace(-1.5, 1.2, 10),
    # np.linspace(-1.5, 1.35, 20),
    np.linspace(-1.5, 1.35, 20),
]

zVecVec = [
    # np.linspace(-1.5, 1.5, 101),
    # np.linspace(-1.5, 1.5, 101),
    # np.linspace(-1.5, 1.5, 101),
    np.linspace(-1.5, 1.5, 101),
]

yStrFormat = [
    # "{:.1f}",
    # "{:.2f}",
    # "{:.2f}",
    "{:.2f}",
]

# print(yVecVec[0])
# print(yVecVec[1])

In [ ]:


fileListVec = []
uxVec = []

spaceAvgUxVec = [] # size: len(folderVec)
spaceAvgUzVec = [] # size: len(folderVec)
for case in range(len(folderVec)):
    # for y in yVec:
    timeAvgUxVecVec = [] # size: len(yVec)
    timeAvgUzVecVec = [] # size: len(yVec)
    for iy in range(0, len(yVecVec[case])):
        yStr = yStrFormat[case].format(yVecVec[case][iy])
        path = f"{folderVec[case]}/probeX{str(x)}Y{yStr}"
        pattern = os.path.join(glob.escape(path), "probe*.txt")
        fileList = glob.glob(pattern)
        print(path, len(fileList))

        fileList.sort()
        # print(pattern)
        #! calculate time averaged U for one Y coord
        timeAvgUxVec = [] # size: len(zVec)
        timeAvgUzVec = [] # size: len(zVec)
        for loc in range(len(fileList)):
            file = fileList[loc]
            step = read_col_probe(file, 0, 2)
            intervalStep = step[1]-step[0]
            idxStart = int(np.abs(step - avgRgVec[case][0]).argmin())
            idxEnd = int(np.abs(step - avgRgVec[case][1]).argmin())
            # print(idxStart, idxEnd)
            ux = read_col_probe(file, 3, 2)
            uz = read_col_probe(file, 4, 2)
            timeAvgUx = np.mean(ux[idxStart:idxEnd]) / U0
            timeAvgUz = np.mean(uz[idxStart:idxEnd]) / U0
            timeAvgUxVec.append(timeAvgUx)
            timeAvgUzVec.append(timeAvgUz)
        print(idxStart, idxEnd)

        timeAvgUxVecVec.append(timeAvgUxVec)
        timeAvgUzVecVec.append(timeAvgUzVec)
    #! calculate space averaged U
    spaceAvgUxVec.append(calculate_average(timeAvgUxVecVec))
    spaceAvgUzVec.append(calculate_average(timeAvgUzVecVec))

refUPath = f"/Users/maggie/repo/scripts/cases/flow_past_cylinder_Re3900/ref/Jacob-x{str(x)}-u.csv"
refVPath = f"/Users/maggie/repo/scripts/cases/flow_past_cylinder_Re3900/ref/Jacob-x{str(x)}-v.csv"
refUX = read_col_csv(refUPath, 0, 1)
refUU = read_col_csv(refUPath, 1, 1)
refVX = read_col_csv(refVPath, 0, 1)
refVV = read_col_csv(refVPath, 1, 1)


In [ ]:
figUx, axUx = plt.subplots(1, 1, figsize=(7, 4), facecolor='w', edgecolor='w')
figUz, axUz = plt.subplots(1, 1, figsize=(7, 4), facecolor='w', edgecolor='w')

if x == 1.06:
    # uxYlim = [-0.25, 1.5]
    uxYlim = [-0.5, 1.5]
    # uzYlim = [-0.15, 0.15]
    uzYlim = [-0.4, 0.4]
elif x == 1.54:
    uxYlim = [-0.5, 1.5]
    uzYlim = [-0.4, 0.4]
elif x == 2.02:
    uxYlim = [-0.5, 1.5]
    uzYlim = [-0.4, 0.4]

axUx.plot(refUX, refUU, lw=0, marker='o', ms=6, fillstyle='none', label='Parnaudeau et al., PIV', c='k')
axUz.plot(refVX, refVV, lw=0, marker='o', ms=6, fillstyle='none', label='Parnaudeau et al., PIV', c='k')

for case in range(len(folderVec)):
    axUx.plot(zVecVec[case], spaceAvgUxVec[case], label=labelVec[case], c=colorVec[case%len(folderVec)], lw=2.5)


    axUz.plot(zVecVec[case], spaceAvgUzVec[case], label=labelVec[case], c=colorVec[case%len(folderVec)], lw=2.5)

axUx.set_xlabel(r"y/D")
# axUx.set_ylabel(r"$<u>/U_0$")
axUx.set_ylabel(r"$u/U_0$")
axUx.set_xlim([-1.5, 1.5])
axUx.set_ylim(uxYlim)
# axUx.legend(loc='upper right', frameon=False, bbox_to_anchor=(1.03, -0.15), ncol=2)
# axUx.legend(loc='upper right', frameon=False, bbox_to_anchor=(1.0, -0.15), ncol=3, fontsize=10)
axUx.tick_params(width=2.0, axis='both', direction='in')
for spine in axUx.spines.values():
    spine.set_linewidth(2.0)
axUx.text(0.85, 0.08, rf"$x/D={x:.2f}$", 
        transform=axUx.transAxes, 
        fontsize=23,
        horizontalalignment='center')

axUz.set_xlabel(r"y/D")
# axUz.set_ylabel(r"$<w>/U_0$")
axUz.set_ylabel(r"$v/U_0$")
axUz.set_xlim([-1.5, 1.5])
axUz.set_ylim(uzYlim)
# axUz.legend(loc='upper right', frameon=False, bbox_to_anchor=(1.05, -0.15), ncol=2)
# axUz.legend(loc='upper right', frameon=False, bbox_to_anchor=(1.0, -0.15), ncol=3, fontsize=10)
axUz.tick_params(width=2.0, axis='both', direction='in')
for spine in axUz.spines.values():
    spine.set_linewidth(2.0)
axUz.text(0.85, 0.08, rf"$x/D={x:.2f}$", 
        transform=axUz.transAxes, 
        fontsize=23,
        horizontalalignment='center')